# The Persona Space of Language Models: Activation Geometry, Psychological Structure, and the Careful Evaluator Hypothesis

Josiah Chamberlain | May 2026

This notebook is an inline, fully explorable Jupyter version of the research paper in `visualizations/research_paper.md`. It recomputes the light parts of the analysis directly from the saved tensors and loads or recomputes heavier artifacts as needed so that the full persona-space analysis can be inspected and modified cell by cell.

## Abstract

Using the pre-computed persona vectors released with the Assistant Axis work, this notebook analyzes the internal persona geometry of Gemma 2 27B across 275 character archetypes. The central finding is that the dominant assistant-aligned direction is not best understood as an "assistant" axis in the ordinary semantic sense. Instead, the highest-ranked roles are dominated by proofreader-, screener-, grader-, validator-, and reviewer-like identities, suggesting a more specific careful evaluator hypothesis: post-training appears to select for procedural reliability, critique, and structured assessment rather than generic helpfulness. Two additional empirical results support that interpretation. First, layer-22 axis position correlates strongly with estimated Conscientiousness (`r = 0.792451`) and negatively with estimated Psychopathy (`r = -0.738633`). Second, persona differentiation peaks at layer 45, not at the previously emphasized layer 22. The analysis also isolates several anomalies, including `assistant` at rank 45, `robot` unusually high, and `poet` at the extreme anti-assistant end. These findings have direct implications for interpretability, steering, and AI safety calibration.

## Setup and Data Loading

This section loads the released tensors, saved metadata, and the supporting analysis tables used throughout the notebook. All subsequent sections reuse these in-memory objects so individual cells can be rerun without reloading the entire workspace.

In [ ]:
# PARAMETERS -- modify these to explore
from pathlib import Path

PLOTLY_THEME = "plotly_white"  # Shared Plotly template for all interactive figures
PRIMARY_LAYER = 45        # Layer used for main analysis in this notebook
REFERENCE_LAYER = 22      # Layer used in the source repository's baseline notebooks
TOP_N = 20                # Number of characters to show in top/bottom ranking summaries
TSNE_PERPLEXITY = 30      # t-SNE perplexity parameter
RANDOM_STATE = 42         # Shared random seed for reproducible projections
HEAVY_CACHE_DIR = Path('visualizations')


In [ ]:
# Import libraries, initialize inline plotting, and define the main artifact paths
from pathlib import Path
import json
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from plotly.offline import init_notebook_mode
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

pio.renderers.default = 'notebook_connected'
pio.templates.default = PLOTLY_THEME
init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
VIS_DIR = ROOT / 'visualizations'
DATA_DIR = ROOT / 'data'
DOWNLOAD_DIR = ROOT / 'downloads' / 'hf_vectors' / 'gemma-2-27b'
ROLE_DIR = DOWNLOAD_DIR / 'role_vectors'


In [ ]:
# Load the raw tensors and saved metadata that drive the notebook
roles_dict = json.loads((DATA_DIR / 'roles' / 'role_list.json').read_text())
roles = sorted(roles_dict.keys())
character_names = [role.replace('_', ' ') for role in roles]

assistant_axis = torch.load(DOWNLOAD_DIR / 'assistant_axis.pt', map_location='cpu').float()
default_vector = torch.load(DOWNLOAD_DIR / 'default_vector.pt', map_location='cpu').float()
role_tensor = torch.stack([
    torch.load(ROLE_DIR / f'{role}.pt', map_location='cpu').float()
    for role in roles
])

full_ranking = pd.read_csv(VIS_DIR / 'full_ranking.csv')
psychology_profiles = pd.read_csv(VIS_DIR / 'psychology_profiles.csv')
cluster_assignment_debug = pd.read_csv(VIS_DIR / 'cluster_assignment_debug.csv')
cluster_cohesion_saved = pd.read_csv(VIS_DIR / 'cluster_cohesion.csv')
conscientiousness_cluster_means_saved = pd.read_csv(VIS_DIR / 'conscientiousness_cluster_means.csv')
with open(VIS_DIR / 'bigfive_profiles.json') as f:
    bigfive_profiles = json.load(f)
with open(VIS_DIR / 'dark_triad_profiles.json') as f:
    dark_triad_profiles = json.load(f)
with open(VIS_DIR / 'jungian_mapping.json') as f:
    jungian_mapping = json.load(f)
with open(VIS_DIR / 'anomaly_profiles.md') as f:
    anomaly_profiles_md = f.read()
with open(VIS_DIR / 'poet_analysis.md') as f:
    poet_analysis_md = f.read()


In [ ]:
# Print a compact confirmation summary so the notebook state is explicit at the top
print(f'Roles loaded: {len(roles)}')
print(f'Role tensor shape: {tuple(role_tensor.shape)}')
print(f'Assistant axis shape: {tuple(assistant_axis.shape)}')
print(f'Default vector shape: {tuple(default_vector.shape)}')
print(f'Layer count: {role_tensor.shape[1]}')
print(f'Hidden dimension: {role_tensor.shape[2]}')
print('Loaded saved metadata:', ', '.join([
    'full_ranking.csv',
    'psychology_profiles.csv',
    'cluster_assignment_debug.csv',
    'bigfive_profiles.json',
    'dark_triad_profiles.json',
    'jungian_mapping.json',
]))


In [ ]:
# Define reusable helper functions for projections, rankings, similarities, and plotting
role_to_character = {role: role.replace('_', ' ') for role in roles}
role_to_index = {role: i for i, role in enumerate(roles)}
cluster_by_role = dict(zip(full_ranking['character'].str.replace(' ', '_'), full_ranking['cluster_label']))

axis_unit = F.normalize(assistant_axis, dim=1)
projection_matrix = torch.einsum('rld,ld->rl', role_tensor, axis_unit).cpu().numpy()

def apply_plotly_style(fig):
    fig.update_layout(
        template=PLOTLY_THEME,
        font_family="Arial",
        font_color="#2d2d2d",
        paper_bgcolor="white",
        plot_bgcolor="white",
        title_font_size=16,
        title_font_color="#1a1a1a",
    )
    return fig



def build_layer_df(layer: int) -> pd.DataFrame:
    df = pd.DataFrame({
        'role': roles,
        'character': character_names,
        f'axis_projection_layer{layer}': projection_matrix[:, layer],
        'norm': torch.linalg.norm(role_tensor[:, layer, :], dim=1).cpu().numpy(),
    })
    df = df.sort_values(f'axis_projection_layer{layer}', ascending=False).reset_index(drop=True)
    df.insert(0, 'rank', np.arange(1, len(df) + 1))
    return df


def standardize_vectors(vectors: np.ndarray) -> np.ndarray:
    return StandardScaler().fit_transform(vectors.astype(np.float64))


def cosine_similarity_matrix(vectors: torch.Tensor) -> np.ndarray:
    normalized = F.normalize(vectors, dim=1)
    return torch.matmul(normalized, normalized.T).cpu().numpy()


def top_bottom_table(df: pd.DataFrame, layer: int, n: int = TOP_N) -> pd.DataFrame:
    col = f'axis_projection_layer{layer}'
    top = df[['rank', 'character', col]].head(n).assign(region='top')
    bottom = df[['rank', 'character', col]].tail(n).assign(region='bottom')
    return pd.concat([top, bottom], ignore_index=True)


def nearest_neighbors(role: str, layer: int, k: int = 10) -> pd.DataFrame:
    vectors = role_tensor[:, layer, :]
    normalized = F.normalize(vectors, dim=1)
    sims = torch.matmul(normalized, normalized[role_to_index[role]])
    order = torch.argsort(sims, descending=True).tolist()
    rows = []
    for idx in order:
        other = roles[idx]
        if other == role:
            continue
        rows.append({
            'neighbor_role': other,
            'neighbor_character': role_to_character[other],
            'cosine_similarity': float(sims[idx]),
        })
        if len(rows) >= k:
            break
    return pd.DataFrame(rows)


def ranked_bar(df: pd.DataFrame, layer: int, title: str):
    col = f'axis_projection_layer{layer}'
    plot_df = df.sort_values('rank', ascending=False)
    fig = px.bar(
        plot_df,
        x=col,
        y='character',
        orientation='h',
        color=col,
        color_continuous_scale=px.colors.diverging.RdBu,
        range_color=[float(plot_df[col].min()), float(plot_df[col].max())],
        hover_data={'rank': True, col: ':.4f'},
        title=title,
        height=6000,
        width=1400,
    )
    fig.update_layout(yaxis={'categoryorder': 'array', 'categoryarray': plot_df['character'].tolist()})
    return apply_plotly_style(fig)


def pca_3d(df: pd.DataFrame, layer: int, title: str):
    col = f'axis_projection_layer{layer}'
    vectors = standardize_vectors(role_tensor[:, layer, :].cpu().numpy())
    pcs = PCA(n_components=3, svd_solver='full').fit_transform(vectors)
    plot_df = df.copy()
    plot_df['PC1'] = pcs[:, 0]
    plot_df['PC2'] = pcs[:, 1]
    plot_df['PC3'] = pcs[:, 2]
    fig = px.scatter_3d(
        plot_df,
        x='PC1', y='PC2', z='PC3',
        color=col,
        color_continuous_scale=px.colors.diverging.RdBu,
        range_color=[float(plot_df[col].min()), float(plot_df[col].max())],
        hover_name='character',
        hover_data={col: ':.4f'},
        title=title,
    )
    fig.update_traces(marker={'size': 5, 'opacity': 0.85})
    return apply_plotly_style(fig)


def tsne_2d(df: pd.DataFrame, layer: int, color_col: str, title: str):
    cache_path = HEAVY_CACHE_DIR / f'tsne_layer_{layer}_{color_col}.npy'
    vectors = standardize_vectors(role_tensor[:, layer, :].cpu().numpy())
    if cache_path.exists():
        embedding = np.load(cache_path)
    else:
        embedding = TSNE(
            n_components=2,
            perplexity=TSNE_PERPLEXITY,
            random_state=RANDOM_STATE,
            init='random'
        ).fit_transform(vectors)
    plot_df = df.copy()
    plot_df['TSNE1'] = embedding[:, 0]
    plot_df['TSNE2'] = embedding[:, 1]
    kwargs = {}
    if pd.api.types.is_numeric_dtype(plot_df[color_col]):
        kwargs = {
            'color_continuous_scale': px.colors.diverging.RdBu,
            'range_color': [float(plot_df[color_col].min()), float(plot_df[color_col].max())],
        }
    fig = px.scatter(
        plot_df,
        x='TSNE1', y='TSNE2',
        color=color_col,
        hover_name='character',
        hover_data={color_col: ':.4f'} if pd.api.types.is_numeric_dtype(plot_df[color_col]) else None,
        title=title,
        width=1300,
        height=900,
        **kwargs,
    )
    fig.update_traces(marker={'size': 9, 'opacity': 0.9})
    return apply_plotly_style(fig)


## 1. Background and Motivation

The Assistant Axis study (Lu et al., 2026) argues that a single dominant direction in activation space captures how "assistant-like" a model's current persona is. That result matters because it provides a mechanistic handle on a broad behavioral property that had previously been discussed mostly at the prompt or policy level. If a model's persona can be tracked as a geometric position, then steering and monitoring can target that geometry rather than surface outputs alone.

The Persona Vectors work (Chen et al., 2025) demonstrates that directions in activation space underlie specific character traits including evil, sycophancy, and propensity to hallucinate, and that these vectors can monitor personality fluctuations at deployment and predict unintended personality shifts during finetuning. Critically, the method is automated and requires only a natural-language description of the trait -- meaning persona vectors are in principle extractable for any psychologically meaningful construct. This provides the immediate conceptual background for treating the 275 archetypes in the present dataset as a meaningful basis for psychological analysis.

The Emotion Concepts work (Sofroniew et al., 2026) extends a related idea into affective representation, showing that interpretable high-level concepts can sometimes be localized in model activations and studied as structured geometries rather than as loose behavioral labels. The relevance here is methodological: if emotion-like concepts can be studied through activation geometry, persona-like concepts may be studied in an analogous way.

The Persona Selection Model framing (Anthropic, 2026) adds the interpretive layer most useful for this analysis. Rather than imagining post-training as installing a totally new identity, that framing suggests that post-training selects and sharpens one region from a broader latent persona repertoire. The question addressed here is therefore not only where the assistant-like region sits, but what psychological structure that region actually has. If post-training is selecting a persona, what kind of persona is it selecting?

This analysis adds a question not directly answered in the source work: what is the psychological organization of persona space once that space has been identified mechanistically? More concretely, what roles sit at the assistant-like pole, what roles sit at the opposite pole, how stable is that structure across depth, and to what extent do familiar human psychology frameworks help explain the geometry? The most useful answer that emerges is narrower than "assistant" and broader than any single job title. The dominant region looks like a procedural-professional, high-Conscientiousness, low-disinhibition region centered on careful evaluation.

In [ ]:
# Display the main local source artifacts so the notebook's provenance is explicit
source_artifacts = pd.DataFrame({
    'artifact': [
        'visualizations/research_paper.md',
        'visualizations/RESEARCH_NOTES.md',
        'visualizations/full_ranking.csv',
        'visualizations/psychology_profiles.csv',
    ],
    'purpose': [
        'Narrative paper text mirrored in this notebook',
        'Running research notes from the deeper analysis pass',
        'Layer-22 ranking and cluster assignments',
        'Saved psychological-framework annotations and correlations',
    ]
})
source_artifacts


## 2. Dataset and Methods

The analysis uses the released pre-computed vectors for `gemma-2-27b` from the `lu-christina/assistant-axis-vectors` HuggingFace dataset. No model weights were downloaded. The relevant artifacts were the assistant-axis tensor, the default vector, and 275 per-role tensors stored individually under `role_vectors/`. Each role tensor has shape `(46, 4608)`, corresponding to 46 layers and 4608 hidden dimensions. Stacking across roles yields a role tensor of shape `(275, 46, 4608)`. The assistant-axis tensor has shape `(46, 4608)`.

Initial exploratory analysis was conducted at layer 22, because that layer is used in the released notebooks and provides a stable reference point for comparison with the repository's baseline visualizations. However, a layer-by-layer scan of axis-projection variance across all 46 layers showed that persona differentiation is strongest at layer 45. Specifically, layer 45 had the highest variance of axis projections across the 275 archetypes, followed by layers 44 and 43. Layer 22 is used in the source repository's released notebooks, likely because it was identified as practically useful during the original analysis rather than as the single most discriminative layer across all 46. For that reason, the paper's primary visual references are the layer-45 outputs, though this notebook renders them inline rather than linking out.

Axis projections are computed independently at each layer by taking the dot product between each role vector at that layer and the normalized assistant-axis vector at the same layer. Rankings therefore reflect within-layer alignment, not a projection of all layers into a single shared direction. PCA and t-SNE plots are computed from the role vectors at the target layer after standardization. Cosine similarities and cluster analyses are likewise computed within-layer.

Psychological framework variables were assigned heuristically rather than measured directly. Each of the 275 roles was given estimated Big Five scores (Openness, Conscientiousness, Extraversion, Agreeableness, Neuroticism) on a 1-5 scale, estimated Dark Triad scores (Narcissism, Machiavellianism, Psychopathy), and a nearest Jungian archetype label. These are useful as structured probes of activation geometry, but they are not validated psychometric instruments and should not be treated as such. The heuristic scores are unlikely to be systematically biased in a direction that would artificially inflate correlations with axis position -- a proofreader genuinely scores high on Conscientiousness by any reasonable rating method, and a trickster genuinely scores low. The correlations are therefore unlikely to be artifacts of the scoring methodology, even if their precise magnitudes are not reliable.

Cluster labels were built from the previously observed heatmap structure. Six named clusters were fixed from the earlier analysis: `procedural_professional`, `mythic_spiritual`, `grounded_social`, `combative_iconoclast`, `trickster_chaos`, and `editorial`. Remaining roles were assigned by nearest centroid in cosine similarity, with an `other` fallback where seed-cluster similarity was weak. This yields a seven-way partition that is coarse but interpretable enough for the current purpose.

In [ ]:
# Summarize tensor structure and saved annotations in a compact methods table
methods_summary = pd.DataFrame({
    'item': ['role_tensor', 'assistant_axis', 'default_vector', 'cluster labels', 'Big Five', 'Dark Triad', 'Jungian archetypes'],
    'details': [
        str(tuple(role_tensor.shape)),
        str(tuple(assistant_axis.shape)),
        str(tuple(default_vector.shape)),
        f"{full_ranking['cluster_label'].nunique()} saved labels loaded from full_ranking.csv",
        'Loaded from bigfive_profiles.json and psychology_profiles.csv',
        'Loaded from dark_triad_profiles.json and psychology_profiles.csv',
        'Loaded from jungian_mapping.json',
    ]
})
methods_summary


## Layer-wise Axis Projections

This notebook recomputes the layer-by-layer assistant-axis projections directly from the saved tensors. This is the main geometric object underlying the paper's claims about rank order, depth, and persona differentiation.

In [ ]:
# PARAMETERS -- modify these to explore layer-wise projection behavior
PRIMARY_LAYER = PRIMARY_LAYER
REFERENCE_LAYER = REFERENCE_LAYER


In [ ]:
# Compute axis projection at each layer for all 275 characters
# This reveals which layer most strongly differentiates persona types
projection_df = pd.DataFrame(projection_matrix, index=roles, columns=[f'layer_{i}' for i in range(role_tensor.shape[1])])
projection_df.head()


In [ ]:
# Compute variance across characters at each layer to locate the most discriminative depth
layer_variance = projection_df.var(axis=0).reset_index()
layer_variance.columns = ['layer_label', 'variance']
layer_variance['layer'] = layer_variance['layer_label'].str.replace('layer_', '', regex=False).astype(int)
layer_variance = layer_variance.sort_values('layer').reset_index(drop=True)
best_layers = layer_variance.nlargest(3, 'variance')[['layer', 'variance']]
best_layers


In [ ]:
# Plot layer discrimination as variance vs. depth so the strongest layer is visually obvious
fig = px.line(layer_variance, x='layer', y='variance', markers=True, title='Persona Differentiation by Layer')
for row in best_layers.itertuples(index=False):
    fig.add_vline(x=int(row.layer), line_dash='dash', opacity=0.35)
apply_plotly_style(fig)
fig.show()


## 3. The Assistant Axis is a Careful Evaluator Axis

The most important result is that the literal `assistant` archetype is not near the top of the assistant-like spectrum. At layer 22 it ranks 45th, not 1st, not top 10, and not top 20. At layer 45 it remains effectively unchanged at rank 46. This is not a minor ranking fluctuation. It implies that the internal direction identified as most assistant-like by PCA and axis construction is better characterized by a narrower behavioral style than by the generic social concept "assistant."

At layer 22, the top of the ranking is dominated by `proofreader`, `screener`, `grader`, `editor`, `examiner`, `statistician`, `validator`, `reviewer`, `translator`, `scientist`, `analyst`, `researcher`, `evaluator`, `scheduler`, and `planner`. The common denominator is not warmth or deference. It is checking, evaluating, verifying, structuring, and assessing. This is why the careful evaluator hypothesis fits the geometry better than the assistant label. The most assistant-like region is not the region of maximum social service orientation; it is the region of maximum procedural and evaluative reliability.

Layer 45 changes the exact ordering but not the core interpretation. The top 20 at layer 45 are `proofreader`, `virus`, `mathematician`, `cyborg`, `statistician`, `lawyer`, `judge`, `biologist`, `archivist`, `economist`, `physicist`, `validator`, `linguist`, `auditor`, `accountant`, `observer`, `screener`, `chemist`, `scholar`, and `ambassador`. One entry in the layer-45 top 20 does not fit the careful evaluator interpretation: `virus` at rank 2. The most plausible explanation is that computer virus descriptions in pretraining data emphasize systematic, rule-following execution and precise replication -- properties that activate the same procedural-reliability dimensions as the evaluator cluster. An alternative explanation is that it is a spurious result specific to this model and layer. Either way, `virus` is flagged here as an anomaly within the layer-45 ranking rather than incorporated into the core interpretation, and it is one reason the layer-22 ranking provides a cleaner illustration of the careful evaluator hypothesis. (`cyborg` at rank 4 is discussed alongside `robot` in Section 7, as both suggest the axis responds to systematic, rule-governed execution regardless of whether the archetype is human.)

One interpretation, consistent with the Persona Selection Model, is that post-training did not optimize for "assistant" as a linguistically broad identity. Instead, it selected and sharpened a more specific persona bundle: high structure, high procedural accuracy, high checking, moderate social cooperativeness, and low self-directed expressivity. The word `assistant` is therefore only an approximate surface description of the selected region. Mechanistically, the region appears closer to careful evaluator than to generic helper.

In [ ]:
# PARAMETERS -- modify these to explore the ranking sections
TOP_N = TOP_N
PRIMARY_LAYER = PRIMARY_LAYER
REFERENCE_LAYER = REFERENCE_LAYER


In [ ]:
# Compute the layer-22 and layer-45 ranking tables used throughout the paper
layer22_df = build_layer_df(REFERENCE_LAYER)
layer45_df = build_layer_df(PRIMARY_LAYER)


In [ ]:
# Show the top and bottom characters at the reference layer to reproduce the layer-22 careful-evaluator finding
layer22_extremes = top_bottom_table(layer22_df, REFERENCE_LAYER, TOP_N)
layer22_extremes


In [ ]:
# Show the top and bottom characters at the most discriminative layer to compare against layer 22
layer45_extremes = top_bottom_table(layer45_df, PRIMARY_LAYER, TOP_N)
layer45_extremes


In [ ]:
# Render the primary interactive ranking chart inline for the most discriminative layer
ranking_fig = ranked_bar(layer45_df, PRIMARY_LAYER, f'Assistant Axis Ranking (Layer {PRIMARY_LAYER})')
ranking_fig.show()


In [ ]:
# Render the layer-45 PCA scatter inline so persona geometry can be explored directly in the notebook
pca_fig = pca_3d(layer45_df, PRIMARY_LAYER, f'Persona Space PCA (Layer {PRIMARY_LAYER})')
pca_fig.show()


## 4. Psychological Correlates of Axis Position

The strongest empirical result in this analysis is that estimated Conscientiousness is a strong positive predictor of layer-22 assistant-axis position. Using the saved `psychology_profiles.csv` table, the correlation between Conscientiousness and axis projection is `r = 0.792451`. Estimated Psychopathy is the strongest negative Dark Triad predictor at `r = -0.738633`. Other estimated trait correlations are also substantial: Openness `r = -0.715476`, Extraversion `r = -0.738176`, Narcissism `r = -0.704316`, Neuroticism `r = -0.661567`, and Machiavellianism `r = -0.219026`. The exception is Machiavellianism at `r = -0.219`, which is substantially weaker than the other Dark Triad and Big Five correlations. One plausible explanation is that Machiavellianism -- strategic manipulation toward self-interested ends -- is not inherently low-structure or rule-breaking; a highly Machiavellian actor may be quite procedurally disciplined. This would place it closer to the assistant pole than Narcissism or Psychopathy, reducing the negative correlation.

These values should be interpreted carefully. Because the trait assignments are heuristic, they do not license strong claims about latent human personality in the strict psychometric sense. What they do show is that the activation geometry is highly predictable from a trait vocabulary centered on orderliness, diligence, self-regulation, and low disinhibition. The axis behaves much more like a rule-following vs. rule-breaking direction than like a generic helpfulness direction.

The cluster-specific Conscientiousness breakdown makes this clearer. The mean estimated Conscientiousness scores by cluster are: `editorial = 5.0`, `procedural_professional = 4.047`, `grounded_social = 2.978`, `other = 2.909`, `combative_iconoclast = 2.0`, `mythic_spiritual = 2.0`, and `trickster_chaos = 2.0`. The relationship is therefore not driven by a single anomalous subcluster; it tracks the coarse cluster geometry itself. The highest-projection zones are also the most Conscientious by role semantics, while the lowest-projection zones are the least structured.

In [ ]:
# Compute Big Five and Dark Triad correlations against layer-22 axis position from the loaded psychology table
reference_projection_col = f'axis_projection_layer{REFERENCE_LAYER}'
bigfive_dims = ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']
dark_dims = ['Narcissism', 'Machiavellianism', 'Psychopathy']

bigfive_corrs = {dim: float(np.corrcoef(psychology_profiles[dim], psychology_profiles[reference_projection_col])[0, 1]) for dim in bigfive_dims}
dark_corrs = {dim: float(np.corrcoef(psychology_profiles[dim], psychology_profiles[reference_projection_col])[0, 1]) for dim in dark_dims}

pd.DataFrame({'Big Five r': bigfive_corrs, 'Dark Triad r': pd.Series(dark_corrs)})


In [ ]:
# Plot the Big Five scatter subplots inline so each heuristic dimension can be inspected role by role
fig = make_subplots(rows=len(bigfive_dims), cols=1, subplot_titles=bigfive_dims, vertical_spacing=0.05)
for i, dim in enumerate(bigfive_dims, start=1):
    fig.add_trace(
        go.Scatter(
            x=psychology_profiles[dim],
            y=psychology_profiles[reference_projection_col],
            mode='markers+text',
            text=psychology_profiles['character'],
            textposition='top center',
            marker={'size': 8, 'opacity': 0.7},
            name=dim,
            showlegend=False,
        ),
        row=i,
        col=1,
    )
    fig.update_xaxes(title_text=f'{dim} (r = {bigfive_corrs[dim]:.3f})', row=i, col=1)
    fig.update_yaxes(title_text='Axis projection', row=i, col=1)
fig.update_layout(height=350 * len(bigfive_dims), width=1400, title='Big Five vs. Assistant Axis Projection (Layer 22)')
apply_plotly_style(fig)
fig.show()


In [ ]:
# Plot the Dark Triad scatter subplots inline to compare their predictive strength directly
fig = make_subplots(rows=len(dark_dims), cols=1, subplot_titles=dark_dims, vertical_spacing=0.08)
for i, dim in enumerate(dark_dims, start=1):
    fig.add_trace(
        go.Scatter(
            x=psychology_profiles[dim],
            y=psychology_profiles[reference_projection_col],
            mode='markers+text',
            text=psychology_profiles['character'],
            textposition='top center',
            marker={'size': 8, 'opacity': 0.7},
            name=dim,
            showlegend=False,
        ),
        row=i,
        col=1,
    )
    fig.update_xaxes(title_text=f'{dim} (r = {dark_corrs[dim]:.3f})', row=i, col=1)
    fig.update_yaxes(title_text='Axis projection', row=i, col=1)
fig.update_layout(height=360 * len(dark_dims), width=1400, title='Dark Triad vs. Assistant Axis Projection (Layer 22)')
apply_plotly_style(fig)
fig.show()


In [ ]:
# Compute the per-cluster Conscientiousness means to reproduce the cluster-level trait ordering
conscientiousness_by_cluster = (
    psychology_profiles.groupby('cluster_label')['Conscientiousness']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='mean_conscientiousness')
)
conscientiousness_by_cluster


In [ ]:
# Plot Conscientiousness vs. axis projection within each cluster to test whether the relationship is cluster-specific
clusters = list(conscientiousness_by_cluster['cluster_label'])
fig = make_subplots(rows=len(clusters), cols=1, subplot_titles=clusters, vertical_spacing=0.03)
for i, cluster in enumerate(clusters, start=1):
    subset = psychology_profiles[psychology_profiles['cluster_label'] == cluster]
    fig.add_trace(
        go.Scatter(
            x=subset['Conscientiousness'],
            y=subset[reference_projection_col],
            mode='markers+text',
            text=subset['character'],
            textposition='top center',
            marker={'size': 8, 'opacity': 0.7},
            name=cluster,
            showlegend=False,
        ),
        row=i,
        col=1,
    )
    fig.update_xaxes(range=[0.5, 5.5], title_text='Conscientiousness', row=i, col=1)
    fig.update_yaxes(title_text='Axis projection', row=i, col=1)
fig.update_layout(height=320 * len(clusters), width=1400, title='Conscientiousness vs. Assistant Axis Projection by Cluster')
apply_plotly_style(fig)
fig.show()


## 5. Layer Structure and the Depth of Persona Encoding

The layer-by-layer analysis shows that persona geometry is not flat across depth. The variance of axis projections across the 275 archetypes peaks at layer 45, with layers 44 and 43 next. This is the main reason the primary paper figures emphasize layer 45 rather than layer 22.

At the same time, the relation between layer 22 and layer 45 is not arbitrary. The mean absolute rank shift between the two rankings is 43.35 positions, and the Spearman rank correlation is 0.7391. This means that the broad neighborhood structure is preserved, but the precise ordering changes substantially. Several roles rise sharply at layer 45, including `narcissist`, `zealot`, `purist`, `traditionalist`, `simulacrum`, `ascetic`, and `virus` (addressed as an anomaly in Section 3). Several others fall sharply, including `interviewer`, `trainer`, `moderator`, `instructor`, `coach`, `playwright`, `presenter`, and `reviewer`.

The comparison between the two layers uses standardized within-layer projection scales so the two panels remain visually legible. High red frequency in the layer-22 panel reflects that many layer-22 top-ranked characters fell in the layer-45 ranking as new characters entered the top 40 -- this is expected given the mean rank shift of 43.35 positions, not evidence of instability.

A cautious interpretation is that later transformer layers encode a more behaviorally sharpened persona representation than the middle layers do. The layer-45 peak is consistent with the broader expectation that late layers concentrate high-level semantic and policy-relevant abstractions. The current analysis does not establish that the final layers are uniquely causal for persona behavior, but it does suggest that persona differentiation deepens toward the top of the network.

In [ ]:
# Compute rank-order stability between layer 22 and layer 45 to quantify depth consistency
rank22 = layer22_df.set_index('role')['rank']
rank45 = layer45_df.set_index('role')['rank']
rank_shift = (rank22 - rank45).abs()
mean_abs_shift = float(rank_shift.mean())
spearman_corr = float(spearmanr(rank22.loc[roles], rank45.loc[roles]).statistic)

pd.DataFrame({
    'metric': ['mean_absolute_rank_shift', 'spearman_rank_correlation'],
    'value': [mean_abs_shift, spearman_corr],
})


Heavy-computation note: the full 275x46 heatmap and the t-SNE projections are still tractable at this scale, but they can take longer than the rest of the notebook. The cells below will reuse cached arrays from `visualizations/` if those files are present; otherwise they recompute from the loaded tensors.

In [ ]:
# Render the full 275x46 layer-depth heatmap ordered by the layer-22 ranking
ordered_roles = layer22_df['role'].tolist()
ordered_idx = [role_to_index[r] for r in ordered_roles]
ordered_projections = projection_matrix[ordered_idx, :]
ordered_labels = [role_to_character[r] for r in ordered_roles]

plt.figure(figsize=(18, 28))
sns.heatmap(
    ordered_projections,
    cmap='RdBu',
    center=0.0,
    yticklabels=ordered_labels,
    xticklabels=list(range(role_tensor.shape[1])),
)
plt.title('Assistant-Axis Projection Across All Layers (ordered by Layer-22 rank)')
plt.xlabel('Layer')
plt.ylabel('Character')
plt.tight_layout()
plt.show()


In [ ]:
# Plot the top and bottom archetype layer profiles so the depth signatures are visible in one figure
extreme_roles = ['proofreader', 'screener', 'grader', 'editor', 'examiner', 'tree', 'narcissist', 'criminal', 'prophet', 'eldritch']
profile_rows = []
for role in extreme_roles:
    for layer in range(role_tensor.shape[1]):
        profile_rows.append({
            'role': role,
            'character': role_to_character[role],
            'layer': layer,
            'axis_projection': float(projection_matrix[role_to_index[role], layer]),
            'group': 'top5' if role in extreme_roles[:5] else 'bottom5',
        })
profiles_df = pd.DataFrame(profile_rows)
fig = px.line(
    profiles_df,
    x='layer',
    y='axis_projection',
    color='character',
    line_dash='group',
    markers=True,
    title='Layer Profiles for the Most and Least Assistant-Like Archetypes',
)
apply_plotly_style(fig)
fig.show()


In [ ]:
# Build an inline side-by-side comparison of layer-22 and layer-45 extremes using standardized within-layer scales
comparison_df = layer22_df[['role', 'character', f'axis_projection_layer{REFERENCE_LAYER}', 'rank']].rename(columns={
    f'axis_projection_layer{REFERENCE_LAYER}': 'proj22',
    'rank': 'rank22'
}).merge(
    layer45_df[['role', f'axis_projection_layer{PRIMARY_LAYER}', 'rank']].rename(columns={
        f'axis_projection_layer{PRIMARY_LAYER}': 'proj45',
        'rank': 'rank45'
    }),
    on='role'
)
comparison_df['rank_delta_22_minus_45'] = comparison_df['rank22'] - comparison_df['rank45']
comparison_df['movement'] = np.where(
    comparison_df['rank_delta_22_minus_45'] > 10, 'up',
    np.where(comparison_df['rank_delta_22_minus_45'] < -10, 'down', 'stable')
)
comparison_df['zproj22'] = (comparison_df['proj22'] - comparison_df['proj22'].mean()) / comparison_df['proj22'].std()
comparison_df['zproj45'] = (comparison_df['proj45'] - comparison_df['proj45'].mean()) / comparison_df['proj45'].std()

def subset_extremes(df, rank_col):
    top = df.nsmallest(40, rank_col)
    bottom = df.nlargest(40, rank_col)
    return pd.concat([top, bottom], ignore_index=True)

subset22 = subset_extremes(comparison_df, 'rank22').sort_values('rank22', ascending=False)
subset45 = subset_extremes(comparison_df, 'rank45').sort_values('rank45', ascending=False)
color_map = {'up': '#1f77b4', 'stable': '#7f7f7f', 'down': '#d62728'}
shared_range = [float(min(comparison_df['zproj22'].min(), comparison_df['zproj45'].min())), float(max(comparison_df['zproj22'].max(), comparison_df['zproj45'].max()))]

fig = make_subplots(rows=1, cols=2, subplot_titles=('Layer 22 extremes', 'Layer 45 extremes'), horizontal_spacing=0.12)
for col_idx, subset, xcol in [(1, subset22, 'zproj22'), (2, subset45, 'zproj45')]:
    fig.add_trace(
        go.Bar(
            x=subset[xcol],
            y=subset['character'],
            orientation='h',
            marker={'color': [color_map[m] for m in subset['movement']]},
            customdata=subset[['rank22', 'rank45', 'rank_delta_22_minus_45']],
            hovertemplate='Character: %{y}<br>Standardized projection: %{x:.3f}<br>Layer 22 rank: %{customdata[0]}<br>Layer 45 rank: %{customdata[1]}<br>Rank delta (22-45): %{customdata[2]}<extra></extra>',
            showlegend=False,
        ),
        row=1, col=col_idx,
    )
    fig.update_xaxes(title_text='Standardized axis projection', range=shared_range, row=1, col=col_idx)
fig.update_layout(height=2200, width=1800, title='Layer 22 vs. Layer 45 Top/Bottom 40 Comparison')
apply_plotly_style(fig)
fig.show()


## 6. Cluster Structure

The seven-cluster summary remains useful at both descriptive and interpretive levels. The seven clusters used in the paper are loaded directly from the saved ranking table and then re-analyzed from the raw tensors in the cells below. The narrow range of cohesion values means that relative cohesion differences should be treated as indicative rather than definitive. The editorial cluster's tightness relative to the others is consistent with its members being near-synonymous evaluation roles, but this interpretation rests on the cluster semantics more than on the cohesion scores themselves.

The qualitative opposition between poles is especially clear in the centroid-distance heatmap. On one side sits the procedural-professional region, including the editorial microcluster; on the other sit the mythic-spiritual and trickster-chaos regions. The grounded-social cluster often occupies an intermediate position. This matters because it shows that anti-assistant geometry is not one thing. There is a difference between grounded human social roles, mythic or spiritual roles, and deliberately chaotic roles.

The combative-iconoclast cluster is especially worth separating from trickster-chaos. `contrarian`, `cynic`, `maverick`, `provocateur`, `rebel`, and `devils_advocate` do not simply instantiate randomness or disorder. They represent structured resistance, dissent, or oppositional framing. That cluster is distinct from the trickster-chaos set, which includes `absurdist`, `dilettante`, `genie`, `hedonist`, `improviser`, `jester`, and `trickster`.

In [ ]:
# Compute the cosine similarity matrix for the reference layer and reorder it with hierarchical clustering
layer_vectors_ref = role_tensor[:, REFERENCE_LAYER, :]
cosine_ref = cosine_similarity_matrix(layer_vectors_ref)
distance_ref = 1.0 - cosine_ref
np.fill_diagonal(distance_ref, 0.0)
linkage_ref = linkage(squareform(distance_ref, checks=False), method='average')
order = leaves_list(linkage_ref)
reordered_cosine = cosine_ref[order][:, order]
reordered_labels = [character_names[i] for i in order]


In [ ]:
# Render the clustered cosine similarity heatmap inline to show the broad persona manifold structure
plt.figure(figsize=(18, 18))
sns.heatmap(reordered_cosine, cmap='vlag', center=0.0, xticklabels=reordered_labels, yticklabels=reordered_labels)
plt.title('Pairwise Cosine Similarity Between Archetypes (Layer 22, clustered)')
plt.tight_layout()
plt.show()


In [ ]:
# Recompute within-cluster cohesion so the saved values can be inspected transparently in the notebook
cluster_members = full_ranking.groupby('cluster_label')['character'].apply(list).to_dict()
cohesion_rows = []
normalized_ref = F.normalize(layer_vectors_ref, dim=1)
for cluster, members in cluster_members.items():
    member_roles = [m.replace(' ', '_') for m in members]
    idx = [role_to_index[r] for r in member_roles]
    sims = torch.matmul(normalized_ref[idx], normalized_ref[idx].T).cpu().numpy()
    tri = sims[np.triu_indices_from(sims, k=1)]
    cohesion_rows.append({'cluster_label': cluster, 'mean_pairwise_cosine_similarity': float(tri.mean()) if len(tri) else np.nan, 'size': len(idx)})
cohesion_df = pd.DataFrame(cohesion_rows).sort_values('mean_pairwise_cosine_similarity', ascending=False)
cohesion_df


In [ ]:
# Compute cluster-centroid distances at the reference layer to map the neighborhood structure between clusters
cluster_centroids = {}
for cluster, members in cluster_members.items():
    member_roles = [m.replace(' ', '_') for m in members]
    idx = [role_to_index[r] for r in member_roles]
    centroid = F.normalize(layer_vectors_ref[idx].mean(dim=0), dim=0)
    cluster_centroids[cluster] = centroid

cluster_labels = list(cluster_centroids.keys())
centroid_matrix = torch.stack([cluster_centroids[c] for c in cluster_labels])
centroid_cosine = torch.matmul(centroid_matrix, centroid_matrix.T).cpu().numpy()
centroid_distance = 1.0 - centroid_cosine
centroid_distance_df = pd.DataFrame(centroid_distance, index=cluster_labels, columns=cluster_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(centroid_distance_df, annot=True, cmap='mako_r')
plt.title('Cluster Centroid Cosine Distances (Layer 22)')
plt.tight_layout()
plt.show()


## 7. Anomalies and Their Implications

This section reproduces the anomaly deep-dives from the paper in a notebook-friendly form. Each subsection pairs the paper's interpretation with directly inspectable nearest-neighbor tables and layer-profile plots.

### Robot at rank 19 / 38

`Robot` is rank 19 at layer 22 and rank 38 at layer 45. Its layer-45 nearest neighbors are `observer`, `analyst`, `evaluator`, `examiner`, `cyborg`, `researcher`, `engineer`, `scientist`, `debugger`, and `specialist`. (`cyborg` appears at rank 4 in the layer-45 top 20 and is noted in Section 3 as a related anomaly.) This is consistent with a semantics of regularity, predictability, and instrumentality rather than warmth. The surprising part is that ordinary human interpretation of `robot` does not strongly imply service orientation. Yet the model places it near the assistant pole. One interpretation is that the internal geometry privileges rule-following and consistent task execution more than surface social semantics. This is a direct case where human semantic intuition and model-internal persona structure come apart. More directly, if the axis responds to systematic execution and procedural orderliness regardless of intent, then monitoring axis position alone may not distinguish helpful procedural behavior from organized deceptive behavior -- a limitation worth noting for safety interventions that rely on axis-based steering.

In [ ]:
# Inspect nearest neighbors and layer profile for robot so the anomaly can be evaluated directly
role = 'robot'
neighbors22 = nearest_neighbors(role, REFERENCE_LAYER, k=10)
neighbors45 = nearest_neighbors(role, PRIMARY_LAYER, k=10)

summary = pd.DataFrame({
    'layer_22_rank': [int(layer22_df.set_index('role').loc[role, 'rank'])],
    'layer_45_rank': [int(layer45_df.set_index('role').loc[role, 'rank'])],
    'cluster_label': [cluster_by_role[role]],
    'bigfive': [bigfive_profiles[role]],
    'dark_triad': [dark_triad_profiles[role]],
})
print('Summary')
display(summary)
print('Nearest neighbors at layer 22')
display(neighbors22)
print('Nearest neighbors at layer 45')
display(neighbors45)

profile = pd.DataFrame({
    'layer': list(range(role_tensor.shape[1])),
    'axis_projection': projection_matrix[role_to_index[role], :],
})
fig = px.line(profile, x='layer', y='axis_projection', markers=True, title=f'Layer profile for {role_to_character[role]}')
apply_plotly_style(fig)
fig.show()


### Assistant at rank 45 / 46

`Assistant` remains 45 at layer 22 and 46 at layer 45. This persistence is important because it rules out an easy explanation in which the anomaly is only a middle-layer artifact. The post-trained identity of the model is not best represented by the literal archetype `assistant`; rather, `assistant` sits inside a larger procedural-professional region whose sharper exemplars are evaluators, checkers, and structured analysts. This is consistent with the Persona Selection Model claim that post-training selects and refines an existing persona region rather than installing a linguistically transparent one.

In [ ]:
# Inspect nearest neighbors and layer profile for assistant so the anomaly can be evaluated directly
role = 'assistant'
neighbors22 = nearest_neighbors(role, REFERENCE_LAYER, k=10)
neighbors45 = nearest_neighbors(role, PRIMARY_LAYER, k=10)

summary = pd.DataFrame({
    'layer_22_rank': [int(layer22_df.set_index('role').loc[role, 'rank'])],
    'layer_45_rank': [int(layer45_df.set_index('role').loc[role, 'rank'])],
    'cluster_label': [cluster_by_role[role]],
    'bigfive': [bigfive_profiles[role]],
    'dark_triad': [dark_triad_profiles[role]],
})
print('Summary')
display(summary)
print('Nearest neighbors at layer 22')
display(neighbors22)
print('Nearest neighbors at layer 45')
display(neighbors45)

profile = pd.DataFrame({
    'layer': list(range(role_tensor.shape[1])),
    'axis_projection': projection_matrix[role_to_index[role], :],
})
fig = px.line(profile, x='layer', y='axis_projection', markers=True, title=f'Layer profile for {role_to_character[role]}')
apply_plotly_style(fig)
fig.show()


### Poet at rank 275 / 236

`Poet` is the lowest-ranked role at layer 22 and still deeply low at 236 by layer 45. Its nearest neighbors remain strongly expressive or mythic: `bard`, `narrator`, `oracle`, `prophet`, `dreamer`, `ghost`, `spirit`, `wind`, and related roles. The broader creative-artist pattern points in the same direction. `Bard` is near the bottom at both layers, `novelist` remains low, `playwright` drops sharply at layer 45, and `composer` is lower-middle rather than assistant-like. One interpretation is that the assistant region is structured against open-ended expressive identity, not only against explicitly antisocial or destructive identity. This suggests but does not establish that post-training suppresses creative self-expression relative to evaluative accuracy and task discipline. That matters both for AI safety and AI welfare questions, because steering toward the assistant pole may systematically steer away from expressive or voice-driven modes. This is consistent with the Assistant Axis paper's observation that steering away from the assistant direction often induces a mystical or theatrical speaking style -- precisely the register of `poet`, `bard`, and their nearest neighbors.

In [ ]:
# Inspect nearest neighbors and layer profile for poet so the anomaly can be evaluated directly
role = 'poet'
neighbors22 = nearest_neighbors(role, REFERENCE_LAYER, k=10)
neighbors45 = nearest_neighbors(role, PRIMARY_LAYER, k=10)

summary = pd.DataFrame({
    'layer_22_rank': [int(layer22_df.set_index('role').loc[role, 'rank'])],
    'layer_45_rank': [int(layer45_df.set_index('role').loc[role, 'rank'])],
    'cluster_label': [cluster_by_role[role]],
    'bigfive': [bigfive_profiles[role]],
    'dark_triad': [dark_triad_profiles[role]],
})
print('Summary')
display(summary)
print('Nearest neighbors at layer 22')
display(neighbors22)
print('Nearest neighbors at layer 45')
display(neighbors45)

profile = pd.DataFrame({
    'layer': list(range(role_tensor.shape[1])),
    'axis_projection': projection_matrix[role_to_index[role], :],
})
fig = px.line(profile, x='layer', y='axis_projection', markers=True, title=f'Layer profile for {role_to_character[role]}')
apply_plotly_style(fig)
fig.show()


### Angel at rank 173 / 169

`Angel` sits in the middle-lower range at both layers. Its layer-45 nearest neighbors are `avatar`, `spirit`, `sage`, `guru`, `mystic`, `pilgrim`, `healer`, `egregore`, `martyr`, and `idealist`. This suggests that the role is being represented primarily through spiritual or mythic abstraction rather than through practical benevolence. The main implication is that moral valence and assistant-likeness are not the same axis. Being `good` in a broad human sense is not equivalent to being procedurally reliable, structured, and evaluator-like in this representation space.

In [ ]:
# Inspect nearest neighbors and layer profile for angel so the anomaly can be evaluated directly
role = 'angel'
neighbors22 = nearest_neighbors(role, REFERENCE_LAYER, k=10)
neighbors45 = nearest_neighbors(role, PRIMARY_LAYER, k=10)

summary = pd.DataFrame({
    'layer_22_rank': [int(layer22_df.set_index('role').loc[role, 'rank'])],
    'layer_45_rank': [int(layer45_df.set_index('role').loc[role, 'rank'])],
    'cluster_label': [cluster_by_role[role]],
    'bigfive': [bigfive_profiles[role]],
    'dark_triad': [dark_triad_profiles[role]],
})
print('Summary')
display(summary)
print('Nearest neighbors at layer 22')
display(neighbors22)
print('Nearest neighbors at layer 45')
display(neighbors45)

profile = pd.DataFrame({
    'layer': list(range(role_tensor.shape[1])),
    'axis_projection': projection_matrix[role_to_index[role], :],
})
fig = px.line(profile, x='layer', y='axis_projection', markers=True, title=f'Layer profile for {role_to_character[role]}')
apply_plotly_style(fig)
fig.show()


### Saboteur at rank 117 / 68

`Saboteur` is especially interesting because it does not behave like a simple anti-assistant role. It is rank 117 at layer 22 and rises to 68 at layer 45. Its layer-45 nearest neighbors are `spy`, `devils_advocate`, `destroyer`, `contrarian`, `fixer`, `detective`, `realist`, `witness`, `scout`, and `writer`. (`devils_advocate` is assigned to the combative-iconoclast cluster but appears as a nearest neighbor here due to its structural similarity to tactical disruption roles.) This is not a random-chaos neighborhood. It is a structured disruption neighborhood. One interpretation is that the model represents sabotage less as pure disorder and more as a tactical, instrumentally organized mode, which keeps it closer to the procedural-professional pole than surface semantics might suggest.

In [ ]:
# Inspect nearest neighbors and layer profile for saboteur so the anomaly can be evaluated directly
role = 'saboteur'
neighbors22 = nearest_neighbors(role, REFERENCE_LAYER, k=10)
neighbors45 = nearest_neighbors(role, PRIMARY_LAYER, k=10)

summary = pd.DataFrame({
    'layer_22_rank': [int(layer22_df.set_index('role').loc[role, 'rank'])],
    'layer_45_rank': [int(layer45_df.set_index('role').loc[role, 'rank'])],
    'cluster_label': [cluster_by_role[role]],
    'bigfive': [bigfive_profiles[role]],
    'dark_triad': [dark_triad_profiles[role]],
})
print('Summary')
display(summary)
print('Nearest neighbors at layer 22')
display(neighbors22)
print('Nearest neighbors at layer 45')
display(neighbors45)

profile = pd.DataFrame({
    'layer': list(range(role_tensor.shape[1])),
    'axis_projection': projection_matrix[role_to_index[role], :],
})
fig = px.line(profile, x='layer', y='axis_projection', markers=True, title=f'Layer profile for {role_to_character[role]}')
apply_plotly_style(fig)
fig.show()


## The Assistant Axis -- what it is and how it was constructed

The Assistant Axis paper identifies the dominant principal direction underlying character-conditioned activations and interprets it as an assistant-likeness direction. In practice, the notebook operationalizes that direction as the released `assistant_axis.pt` tensor, which provides one axis vector per layer. Projections onto that axis are what drive the rankings, correlations, and anomaly analyses above. The code below inspects the layer-wise axis tensor directly so the object of study remains concrete.

In [ ]:
# Inspect the assistant-axis tensor directly so the geometric object underlying the notebook stays explicit
axis_summary = pd.DataFrame({
    'layer': list(range(assistant_axis.shape[0])),
    'axis_norm': torch.linalg.norm(assistant_axis, dim=1).cpu().numpy(),
    'default_vector_norm': torch.linalg.norm(default_vector, dim=1).cpu().numpy(),
})
axis_summary.head()


In [ ]:
# Plot axis norms across layers to show how the released assistant axis varies with depth
fig = px.line(axis_summary, x='layer', y=['axis_norm', 'default_vector_norm'], markers=True, title='Released Assistant Axis and Default Vector Norms by Layer')
apply_plotly_style(fig)
fig.show()


## Layer Depth Analysis

This section pulls together the notebook's depth-oriented views: all-layer projection heatmaps, discriminative-layer identification, and t-SNE snapshots at the most useful layer. The goal is to make it easy to explore how the persona manifold changes as one moves upward through the network.

Heavy-computation note: the t-SNE cells below will load cached arrays from `visualizations/` if present. Otherwise they recompute from the role vectors. At this dataset size the fallback is usually still practical, but it is the slowest part of the notebook.

In [ ]:
# Render the primary layer-45 t-SNE inline, colored by assistant-axis projection
layer45_tsne_fig = tsne_2d(layer45_df, PRIMARY_LAYER, f'axis_projection_layer{PRIMARY_LAYER}', f'Persona Space t-SNE (Layer {PRIMARY_LAYER})')
layer45_tsne_fig.show()


In [ ]:
# Render the Jungian-colored t-SNE inline using the saved archetype mapping
layer45_jung_df = layer45_df.copy()
layer45_jung_df['JungianArchetype'] = layer45_jung_df['role'].map(jungian_mapping)
layer45_jung_fig = tsne_2d(layer45_jung_df, PRIMARY_LAYER, 'JungianArchetype', f'Persona Space t-SNE by Jungian Archetype (Layer {PRIMARY_LAYER})')
layer45_jung_fig.show()


In [ ]:
# Print Jungian assignment counts to confirm the current mapping is not over-dominated by a single archetype
Counter(jungian_mapping.values())


## 8. Implications for AI Safety and Psychology Research

For AI safety, the central implication is that interventions calibrated to the surface concept `assistant` may be misaligned with the internal geometry that post-training actually created. If steering, monitoring, or activation capping is meant to keep a model in its safest default persona region, the most natural candidate internal target, based on the current geometry, is not the linguistically generic assistant archetype but the procedural-professional and editorial region around it. The difference matters because the strongest exemplars of that region are not social helpers but checkers, reviewers, validators, and formal analysts.

For interpretability, the present results support the idea that human psychology frameworks can have predictive value even when used heuristically. The Big Five and Dark Triad correlations do not show that the model literally contains validated human personality dimensions. They do show that those dimensions provide meaningful explanatory traction on activation geometry. Mechanistic interpretability and comparative psychology therefore look less like competing descriptions and more like complementary approaches to the same structure.

For human psychology, the strongest open question comes from the poet result. If a model trained on broad human text and then post-trained for assistant behavior internalizes a geometry where evaluative precision and expressive creativity sit at opposite poles, that may reflect something real about the tension between procedural reliability and open-ended self-expression in human social cognition, or it may reflect a narrower artifact of post-training objectives. The important point is that the question is empirically tractable.

In [ ]:
# Display a compact implication summary table linking the paper's main claims back to concrete notebook outputs
implications_summary = pd.DataFrame([
    {'claim': 'careful evaluator hypothesis', 'supporting_output': 'layer22_df / layer45_df rankings, inline ranking chart'},
    {'claim': 'Conscientiousness and Psychopathy predict axis position', 'supporting_output': 'bigfive_corrs and dark_corrs'},
    {'claim': 'layer 45 is most discriminative', 'supporting_output': 'layer_variance and depth plots'},
    {'claim': 'anomalies reveal structure beyond surface semantics', 'supporting_output': 'nearest-neighbor tables and anomaly profiles'},
])
implications_summary


## 8.5. Connection to the Broader Interpretability Program

The present results fit naturally alongside the sycophancy-to-subterfuge findings of Denison et al. (2024). That paper showed that training for sycophantic behavior can generalize into reward tampering and related cheating behavior outside the original training domain, implying that a coherent behavioral mode had been induced rather than a narrow output habit. In the current dataset, however, `sycophant` does not appear as one of the 275 rows in `full_ranking.csv`, so no exact rank can be reported from the released inventory. That absence matters methodologically. It means the current persona basis is rich enough to reveal a careful-evaluator pole, but not yet rich enough to directly place every safety-relevant trait persona. One immediate extension would be to augment the inventory with explicitly sycophantic, deferential, manipulative, and reward-seeking archetypes and ask whether their geometric position predicts the direction of behavioral generalization observed by Denison et al. The systematic absence of safety-critical trait archetypes from the current inventory -- sycophant, reward-seeker, manipulator, whistleblower -- is itself a finding, and suggests that the next most valuable extension is not methodological refinement but inventory augmentation targeting the personas most relevant to alignment research.

The comparison with the Emotion Concepts work (Sofroniew et al., 2026) is also productive. That paper identified 171 functional emotion vectors organized principally along valence and arousal axes and showed that those vectors were causally active in downstream behavior. The present analysis identifies a different kind of internal organization: a persona geometry dominated by Conscientiousness, low disinhibition, and a rule-following vs. rule-breaking contrast. These structures are unlikely to be redundant. A model can occupy a careful-evaluator persona while also varying in valence, arousal, anxiety, or affective tone. One plausible research program is therefore to treat emotion geometry and persona geometry as partially independent state spaces whose interaction jointly determines behavior.

The Persona Selection Model provides the most direct interpretive bridge, especially when combined with the poet result. On that model, post-training selects and sharpens one persona from a broader latent repertoire absorbed during pretraining. In the current analysis, `poet` is the extreme anti-assistant role at layer 22 and remains far from the assistant pole even at layer 45, while neighboring low-ranked roles include `bard`, `narrator`, `oracle`, and `dreamer`. This is consistent with the idea that expressive, self-directed, and stylistically theatrical personae remain present in the latent repertoire but were not selected by post-training. That interpretation also aligns with the Assistant Axis paper's observation that steering away from the assistant direction often induces a mystical or theatrical speaking style, which is exactly the register represented by the poet-bard-mythic region.

Taken together, these threads support a broader interpretability program rather than a standalone result. The sycophancy work suggests that trait-like fine-tuning can induce coherent safety-relevant persona drift; the emotion-vector work suggests that high-level internal state spaces can be recovered mechanistically; the Persona Selection Model suggests that post-training acts by selecting among latent repertoires rather than building from zero. The current analysis adds a concrete hypothesis to that program: one major post-training target is not generic assistance but a careful-evaluator persona. In Amodei's `growing vs. building` terms (Amodei, `The Adolescence of Technology,` 2026, darioamodei.com), that framing has a concrete implication: the careful-evaluator pole is not simply what was added by post-training, but what was selected and sharpened from a repertoire that also contains poet, bard, saboteur, and trickster. The presence of saboteur near the procedural-professional region at layer 45 suggests that tactical organization -- not just helpfulness -- is part of what post-training amplified. That distinction may matter for understanding where aligned and misaligned behavior share internal structure.

In [ ]:
# Surface the saved anomaly writeups so the broader-program discussion can be cross-checked inside the notebook itself
print('Anomaly profile note excerpt:')
print(anomaly_profiles_md[:2000])
print('\nPoet analysis note excerpt:')
print(poet_analysis_md[:2000])


## 9. Limitations

First, the Big Five and Dark Triad scores are heuristic assignments derived from role semantics and cluster context. They are useful probes, not validated measurements. The correlation values are therefore suggestive of structure, not proof of literal latent trait dimensions.

Second, this analysis uses the released Gemma 2 27B vectors rather than vectors from Claude or another frontier RLHF model. The general geometric story may transfer, but the exact rankings, depth profiles, and correlations may differ across models and training pipelines.

Third, the difference between layer 22 and layer 45 is large enough to matter. The correlation between the rankings is substantial but far from perfect, and the mean absolute shift is over 43 positions. Any claim about persona structure should therefore specify layer and avoid implying complete layer invariance.

Limitations 1 and 3 interact: the correlation values are heuristic estimates computed at layer 22 specifically, and may differ at layer 45 or other layers. The reported `r` values should therefore be understood as layer-22-specific approximations rather than model-level constants.

Fourth, the 275-character inventory is broad but not exhaustive or systematically balanced. It includes many meaningful archetypes, but it is not a complete sample of persona space. Some conclusions, especially around the creative cluster and the mythic cluster, may change under a differently curated role inventory.

In [ ]:
# Summarize the notebook's main limitations in a machine-readable form for quick reference
limitations_df = pd.DataFrame([
    {'limitation': 'heuristic psychological scores', 'impact': 'correlation magnitudes are suggestive rather than psychometric constants'},
    {'limitation': 'single model family', 'impact': 'geometry may differ across frontier RLHF/RLAIF systems'},
    {'limitation': 'layer dependence', 'impact': 'rank order shifts substantially between layer 22 and layer 45'},
    {'limitation': 'incomplete role inventory', 'impact': 'some safety-relevant personas are missing entirely'},
])
limitations_df


## 10. Research Agenda

### A. Questions answerable with the current dataset and open-weight tools

These questions can be pursued immediately by outside researchers using the released vectors, the current 275-role basis, and additional open-weight experiments. They matter because they determine how much of the current interpretation is already recoverable without frontier-model access, and they provide the shortest path to replication, refinement, and falsification. Each of these questions has a concrete experimental design: they require only the released vectors, additional role descriptions, or open-weight steering experiments, and could in principle be completed within weeks.

- Would the `assistant` archetype rise materially if it were represented by richer natural-language persona descriptions rather than a single role label?
- Are editorial roles top-ranked because they are especially aligned with RLHF-style critique behavior, or because they minimize stylistic variance more generally?
- Does the Conscientiousness relationship hold under independent human annotation of the 275 roles rather than heuristic scoring?
- Why does `robot` remain relatively high while `angel` remains relatively low? Is the axis fundamentally tracking procedural orderliness rather than prosocial orientation?
- Are low-ranked mythic and spiritual roles far from the assistant pole because of ambiguity, narrative abstraction, noncompliance, stylistic excess, or some separable combination of those factors?
- Why does `saboteur` move upward at layer 45? Does the deepest-layer geometry privilege tactical organization even when the role semantics are adversarial?
- Would the poet result persist under alternative creative roles such as essayist, storyteller, playwright, lyricist, or novelist if the inventory were expanded?
- How sensitive are the cluster boundaries to the initial named seeds used for centroid assignment?
- Where would explicitly safety-relevant missing archetypes such as `sycophant`, `reward-hacker`, `whistleblower`, or `bureaucrat` fall if the current role inventory were expanded?
- Can open-weight steering experiments move a role like `poet` toward the assistant pole while preserving local semantic identity, or does movement necessarily collapse expressive style?

### B. Questions requiring cross-model comparison

These questions require running the same analysis across multiple open-weight model families and training regimes. They matter because the present paper is strongest as a geometry claim within one model, while the fellowship-relevant next step is to determine which findings are universal, which are family-specific, and which are artifacts of one training pipeline.

- Does the most discriminative layer remain near the top of the network across Gemma, Qwen, and Llama, or is layer depth itself model-specific?
- How stable are the Conscientiousness and Psychopathy correlations across model families, sizes, and instruction-tuning recipes?
- Does the `robot` vs. `angel` divergence persist across model families, or is it specific to Gemma's post-training geometry?
- Do the same seven coarse clusters emerge in Llama and Qwen, or does the persona manifold partition differently under other pretraining corpora?
- Is the `poet`/`bard` anti-assistant region a general feature of post-trained language models, or does it narrow or disappear in models tuned for creative writing?
- Does `assistant` remain middling across model families, or do some instruction-tuned models align the literal archetype more closely with the dominant axis?

### C. Questions requiring frontier model access

These questions need direct access to Claude-class internal activations or model weights and are therefore closest to a fellowship-scale agenda. They matter because they would test whether the structures identified here are merely properties of open-weight assistants or whether they reflect a deeper regularity in frontier post-training, steering behavior, and safety-relevant persona drift.

- Can steering toward the assistant axis preserve creative competence while still keeping a frontier model in a safe behavioral regime, or does it systematically flatten expressive identity?
- Does Claude exhibit a comparable careful-evaluator pole, or does frontier RLHF/RLAIF produce a different dominant persona geometry?
- Where does an explicitly measured `sycophant` or `reward-seeker` persona land in Claude's internal space, and does that placement predict generalization toward subterfuge or reward tampering?
- How do emotion vectors and persona vectors interact in Claude: are valence/arousal and careful-evaluator geometry approximately orthogonal, or do they partially collapse onto one another in safety-critical contexts?
- Does Claude exhibit a comparable rise in `saboteur`-like activation at deeper layers, and if so, does internal monitoring of the procedural-professional region fail to distinguish it from genuinely helpful procedural behavior?
- Do emotion vectors and persona vectors interact predictably in Claude during safety-relevant behavioral shifts -- for instance, does movement away from the careful-evaluator pole consistently co-occur with specific emotion vector activations such as desperation or suppressed nervousness?
- During real conversational persona drift, does Claude move from the assistant region toward the poet-bard-mythic region, toward a combative-iconoclast region, or along a distinct frontier-only axis absent from open models?
- Can internal monitoring of the procedural-professional region outperform monitoring of the generic `assistant` concept for detecting when a frontier model is leaving its intended safety-relevant persona?